# 04 - Clean the matches
Produces the table tennis_clean.matches
Run after 02_load_tennis_bigquery

## Install the pinned libraries

In [1]:

REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

ERROR: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt


## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
RAW = f"{PROJECT_ID}.tennis_raw.wta_matches"
CLEAN = f"{PROJECT_ID}.tennis_clean.matches"

## Build the typed table
Row-wise changes only so this is safe before the train/test split:
- text converted to numbers and dates with `SAFE_CAST` (empty text becomes NULL, and anything that cannot convert becomes NULL)
- `surface` lower-cased, text trimmed
- scope: levels G, PM, P, I, and no event that has a round-robin stage
- non-numeric seed values (Q, WC, LL) moved to the entry column
- `season` taken from the year in `tourney_id` (the source's own season). A handful of tournaments that start in the last days of December belong to the next season, so their `season` differs from the year of `tourney_date`
- flags for walkovers, retirements and defaults (from the score text)

The bytes processed figure printed at the end indicates cost.

In [ ]:
INT_COLS = ["draw_size", "best_of", "match_num", "winner_id", "winner_seed", "winner_ht",
            "loser_id", "loser_seed", "loser_ht", "minutes",
            "w_ace", "w_df", "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon", "w_SvGms", "w_bpSaved", "w_bpFaced",
            "l_ace", "l_df", "l_svpt", "l_1stIn", "l_1stWon", "l_2ndWon", "l_SvGms", "l_bpSaved", "l_bpFaced",
            "winner_rank", "winner_rank_points", "loser_rank", "loser_rank_points"]
FLOAT_COLS = ["winner_age", "loser_age"]
TEXT_COLS = ["winner_name", "winner_hand", "winner_ioc", "loser_name", "loser_hand", "loser_ioc", "score", "round"]

select_parts = (
    [f"SAFE_CAST(NULLIF(TRIM({c}), '') AS INT64) AS {c}" for c in INT_COLS]
    + [f"SAFE_CAST(NULLIF(TRIM({c}), '') AS FLOAT64) AS {c}" for c in FLOAT_COLS]
    + [f"NULLIF(TRIM({c}), '') AS {c}" for c in TEXT_COLS]
)

# Keep an existing entry value if there is one; otherwise take a non-numeric seed value as the entry code.
# The seed columns still convert with SAFE_CAST, so those seeds stay NULL (those players were not seeded).
def entry_expr(side):
    seed = f"NULLIF(TRIM({side}_seed), '')"
    return f"""COALESCE(
        UPPER(NULLIF(TRIM({side}_entry), '')),
        IF(SAFE_CAST({seed} AS INT64) IS NULL, UPPER({seed}), NULL)
      ) AS {side}_entry"""

select_parts += [entry_expr("winner"), entry_expr("loser")]

sql = f"""
CREATE OR REPLACE TABLE `{CLEAN}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'Typed WTA matches, levels G/PM/P/I, no round-robin events. Built from tennis_raw.wta_matches.')
AS
SELECT
  CONCAT(tourney_id, '-', match_num) AS match_id,
  tourney_id,
  tourney_name,
  LOWER(TRIM(tourney_name)) AS name_key,
  PARSE_DATE('%Y%m%d', tourney_date) AS tourney_date,
  CAST(SUBSTR(tourney_id, 1, 4) AS INT64) AS season,
  tourney_level,
  LOWER(NULLIF(TRIM(surface), '')) AS surface,
  REGEXP_CONTAINS(COALESCE(score, ''), r'W/O') AS is_walkover,
  REGEXP_CONTAINS(COALESCE(score, ''), r'RET') AS is_retirement,
  REGEXP_CONTAINS(COALESCE(score, ''), r'DEF') AS is_default,
  {", ".join(select_parts)}
FROM `{RAW}`
WHERE tourney_level IN ('G', 'PM', 'P', 'I')
  AND tourney_id NOT IN (SELECT DISTINCT tourney_id FROM `{RAW}` WHERE round = 'RR')
"""
job = bq.query(sql)
job.result()
print("Built", CLEAN, "| bytes processed:", f"{job.total_bytes_processed:,}")

## Checks


In [ ]:
# 1. Row count and uniqueness
print(run(f"SELECT COUNT(*) AS rows_total, COUNT(DISTINCT match_id) AS unique_match_ids FROM `{CLEAN}`"))

# 2. Did any numeric-looking value fail to convert? (raw looks like a number, typed table has NULL)
cols = INT_COLS + FLOAT_COLS
NUM = r"^-?[0-9]+(\.[0-9]+)?$"
bad = run(f"""
SELECT {", ".join(f"COUNTIF(REGEXP_CONTAINS(TRIM(r.{c}), r'{NUM}') AND m.{c} IS NULL) AS {c}" for c in cols)},
       COUNT(*) AS joined_rows
FROM `{RAW}` r
JOIN `{CLEAN}` m ON r.tourney_id = m.tourney_id AND SAFE_CAST(r.match_num AS INT64) = m.match_num
""").T
print(bad[bad[0] > 0])     # joined_rows only, if nothing was lost

# 2b. Non-numeric seed values are entry codes: each must now be in the entry column
print(run(f"""
SELECT
  COUNTIF(NULLIF(TRIM(r.winner_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(r.winner_seed) AS INT64) IS NULL) AS winner_codes_in_seed_column,
  COUNTIF(NULLIF(TRIM(r.winner_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(r.winner_seed) AS INT64) IS NULL AND m.winner_entry IS NULL) AS winner_codes_missing_from_entry,
  COUNTIF(NULLIF(TRIM(r.loser_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(r.loser_seed) AS INT64) IS NULL) AS loser_codes_in_seed_column,
  COUNTIF(NULLIF(TRIM(r.loser_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(r.loser_seed) AS INT64) IS NULL AND m.loser_entry IS NULL) AS loser_codes_missing_from_entry
FROM `{RAW}` r
JOIN `{CLEAN}` m ON r.tourney_id = m.tourney_id AND SAFE_CAST(r.match_num AS INT64) = m.match_num
"""))

# 3. Labels, seasons, flags, round robin, entry codes
print(run(f"SELECT tourney_level, surface, COUNT(*) AS matches FROM `{CLEAN}` GROUP BY 1, 2 ORDER BY 1, 2"))
print(run(f"SELECT season, COUNT(*) AS matches FROM `{CLEAN}` GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT COUNTIF(is_walkover) AS walkovers, COUNTIF(is_retirement) AS retirements, COUNTIF(is_default) AS defaults, COUNTIF(round = 'RR') AS round_robin_rows FROM `{CLEAN}`"))
print(run(f"SELECT winner_entry, COUNT(*) AS n FROM `{CLEAN}` GROUP BY 1 ORDER BY n DESC"))

# 4. Tournaments whose season differs from the year of their start date (year-boundary events)
run(f"""SELECT season, EXTRACT(YEAR FROM tourney_date) AS date_year, tourney_name, tourney_date, COUNT(*) AS matches
        FROM `{CLEAN}` WHERE season <> EXTRACT(YEAR FROM tourney_date)
        GROUP BY 1, 2, 3, 4 ORDER BY tourney_date""")